## Import and initialize packages 

In [1]:
import pyrosetta_installer; pyrosetta_installer.install_pyrosetta()
import pyrosetta;
from pyrosetta import *
import glob
from pyrosetta.toolbox import cleanATOM, mutate_residue
from random import randrange,choice
import copy

PyRosetta install detected, doing nothing...


In [2]:
print(glob.glob("./final_darpins/*.pdb"))
pyrosetta.init("-mute all")

['./final_darpins/2.pdb', './final_darpins/1.pdb', './final_darpins/5.pdb', './final_darpins/4.pdb', './final_darpins/3.pdb']
┌──────────────────────────────────────────────────────────────────────────────┐
│                                 PyRosetta-4                                  │
│              Created in JHU by Sergey Lyskov and PyRosetta Team              │
│              (C) Copyright Rosetta Commons Member Institutions               │
│                                                                              │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRE PURCHASE OF A LICENSE │
│         See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└──────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2025 [Rosetta PyRosetta4.Release.python313.ubuntu 2025.16+release.729367d2066d37ea8bbc2d993d207ba805937601 2025-04-14T15:28:27] retrieved from: http://www.pyrosetta.org


## Get the energy of each candidate

In [4]:
AA= ['A', 'R', 'N', 'D', 'C', 'Q', 'E', 'G', 'H', 'I', 'L', 'K', 'M', 'F', 'P', 'S', 'T', 'W', 'Y', 'V']
sfxn = get_score_function()
fr = pyrosetta.rosetta.protocols.relax.FastRelax(scorefxn_in=sfxn , standard_repeats=1)# , standard_repeats=1
# fr.constrain_relax_segments()

for i in range(5):
    ligand_params = Vector1(['./Autoinduced_Peptide_1.params'])
    pose = pyrosetta.Pose()
    res_set = pose.conformation().modifiable_residue_type_set_for_conf()
    res_set.read_files_for_base_residue_types( ligand_params )
    pose.conformation().reset_residue_type_set_for_conf( res_set )
    pose_from_file(pose, "./final_darpins/"+str(i+1)+".pdb")
    fr.apply(pose)
    print(i)
    print(pose.sequence())
    print(sfxn(pose))

0
DLGKKLLEAARAGQDDEVRILMANGADVNAQSSAGHTPLHLAAHWGHLEIVEVLLKNGADVNAADKHGNTPLHLAAKAGHLEIVEVLLKHGADVNAADDNGRTPLHLAAQTGHLEIVEVLLKYGADVNAQDKFGKTPFDLAIDNGNEDIAEVLQKAAZ
-582.5793109300321
1
DLDELLLEAALKGDADLVRKLIKLGADVNTKTEMGWTPLHLAAYYGFLEIVLLLLRNGADANAQDKYGWTPLHLAVLSGQLEIVIVLLLFGADANAYTKDGITPLILAVARNFLEIVILLLRWGADVDTYDKLGRTPLDYARDLGFEEIYRTLLAYKZ
-699.6194229482568
2
ZDLGKKLLEAARAGQDDEVRILMANGADVNAQDSQGWTPLHLAAAYGHLEIVEVLLKNGADVNAADKYGWTPLHLAAYYGHLEIVEVLLKHGADVNAADKWGNTPLHLAAAAGHLEIVEVLLKYGADVNAQDKFGKTPFDLAIDNGNEDIAEVLQKAA
-625.8195515067011
3
ZDLGKKLLEAARAGQDDEVRILMANGADVNAQDSSGWTPLHLAAAYGHLEIVEVLLKNGADVNAADKTGNTPLHLAAAHGHLEIVEVLLKHGADVNAANKDGWTPLHLAAAHGHLEIVEVLLKHGADVNAQDKFGKTPFDLAIDNGNEDIAEVLQKAA
-622.524004147561
4
ZDLGKKLLEAARAGQDDEVRILMANGADVNAQDKDGNTPLHLAARHGHLEIVEVLLKHGADVNAANKTGWTPLHLAAWYGHLEIVEVLLKHGADVNAADKRGWTPLHLAAAAGHLEIVEVLLKYGADVNAQDKFGKTPFDLAIDNGNEDIAEVLQKAA
-622.8610251456307


## Get the average energy of random DARPins (within the constraints)

In [10]:
residue_mut = [[32], [33, 34, 36, 44, 45], [41], [57]]
AA_options = [
    ["D", "N", "S", "T"],
    ["A", "R", "N", "D", "Q", "E", "H", "K", "S", "T", "W", "Y"],
    ["A", "S", "T", "V", "L"],
    ["N", "H", "Y"],
]
all_mutations = [[]]
for i in range(len(residue_mut)):
    for j in range(len(residue_mut[i])):
        for k in range(len(AA_options[i])):
            for l in range(3):
                all_mutations.append([residue_mut[i][j] + 1 + l * 33, AA_options[i][k]])
all_mutations = all_mutations[1:]

ligand_params = Vector1(['./Autoinduced_Peptide_1.params'])
pose = pyrosetta.Pose()
res_set = pose.conformation().modifiable_residue_type_set_for_conf()
res_set.read_files_for_base_residue_types( ligand_params )
pose.conformation().reset_residue_type_set_for_conf( res_set )
pose_from_file(pose, "./final_darpins/3.pdb")

from tqdm import tqdm
scores=[]
mut_done=[]
for i in range(10):
    for j in range(10000):  # terrible way to randomize mut positions
        mut = choice(all_mutations)
        if mut[0] in mut_done:
            continue
        mut_done.append(mut[0])
        mutate_residue(pose, mut[0], mut[1])  # 1-indexed AA to mutate
    fr.apply(pose)
    scores.append(sfxn(pose))
    print(scores[-1])
print(sum(scores)/len(scores))

-564.0249320568698
-560.6425370026634
-565.2752353916793
-566.3696699696484
-555.8288339068331
-565.7429192879536
-566.573834082729
-566.6127016138947
-566.3938815466536
-568.7522987722782
-564.6216843631203
